In [1]:
pip install pandas numpy matplotlib openpyxl


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns
pd.set_option('display.max_columns', None)

# Display wider tables
pd.set_option('display.width', None)

# Display numbers without scientific notation
pd.options.display.float_format = '{:.2f}'.format

### Load dataset


In [6]:
from pathlib import Path

# Project root
project_root = Path.cwd().parent

# Dataset path
file_path = project_root / "data" / "raw" / "Online Retail.xlsx"

# Read dataset
df = pd.read_excel(file_path)

# Display first 5 rows
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


In [8]:
df.shape

(541909, 8)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [12]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [14]:
df.duplicated().sum()

5268

In [16]:
df["CustomerID"].nunique()

4372

In [18]:
df["StockCode"].nunique()

4070

In [20]:
df["Country"].nunique()

38

In [22]:
print(df["InvoiceDate"].min())

print(df["InvoiceDate"].max())

2010-12-01 08:26:00
2011-12-09 12:50:00


In [24]:
df[df["InvoiceNo"].astype(str).str.startswith("C")].shape

(9288, 8)

In [26]:
df[df["Quantity"] < 0].shape

(10624, 8)

In [28]:
df[df["UnitPrice"] == 0].shape

(2515, 8)

In [30]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.00,541909,541909.00,406829.00
mean,9.55,2011-07-04 13:34:57.156386048,4.61,15287.69
min,-80995.00,2010-12-01 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-28 11:34:00,1.25,13953.00
50%,3.00,2011-07-19 17:17:00,2.08,15152.00
75%,10.00,2011-10-19 11:27:00,4.13,16791.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


In [32]:
# Total cancelled rows
cancelled = df[df["InvoiceNo"].astype(str).str.startswith("C")]

print("Cancelled Rows:", len(cancelled))

# Cancelled rows with negative quantity
cancelled_negative = cancelled[cancelled["Quantity"] < 0]

print("Cancelled + Negative Quantity:", len(cancelled_negative))

# Cancelled rows with positive quantity
cancelled_positive = cancelled[cancelled["Quantity"] > 0]

print("Cancelled + Positive Quantity:", len(cancelled_positive))

Cancelled Rows: 9288
Cancelled + Negative Quantity: 9288
Cancelled + Positive Quantity: 0


In [34]:
negative = df[df["Quantity"] < 0]

negative.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.00,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.00,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.00,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.00,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.00,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.00,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.00,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.00,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.00,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.00,United Kingdom


In [36]:
missing_customer = df[df["CustomerID"].isnull()]

missing_customer.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom


In [38]:
duplicates = df[df.duplicated()]

duplicates.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.00,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.00,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.00,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.00,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.00,United Kingdom
587,536412,22273,FELTCRAFT DOLL MOLLY,1,2010-12-01 11:49:00,2.95,17920.00,United Kingdom
589,536412,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-01 11:49:00,3.75,17920.00,United Kingdom
594,536412,22141,CHRISTMAS CRAFT TREE TOP ANGEL,1,2010-12-01 11:49:00,2.10,17920.00,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.00,United Kingdom
600,536412,22569,FELTCRAFT CUSHION BUTTERFLY,2,2010-12-01 11:49:00,3.75,17920.00,United Kingdom


In [40]:
zero_price = df[df["UnitPrice"] == 0]

zero_price.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.00,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,NaN,United Kingdom


In [42]:
df.groupby("InvoiceNo").size().sort_values(ascending=False).head(10)

InvoiceNo
573585    1114
581219     749
581492     731
580729     721
558475     705
579777     687
581217     676
537434     675
580730     662
538071     652
dtype: int64

In [44]:
customer_orders = df.groupby("CustomerID")["InvoiceNo"].nunique()

customer_orders.describe()

count   4372.00
mean       5.08
std        9.34
min        1.00
25%        1.00
50%        3.00
75%        5.00
max      248.00
Name: InvoiceNo, dtype: float64

# PHASE 2 - Data Cleaning & Preparation

In [47]:
clean_df = df.copy()

print(clean_df.shape)

(541909, 8)


In [49]:
print("Before:", clean_df.shape)

clean_df = clean_df.drop_duplicates()

print("After:", clean_df.shape)

Before: (541909, 8)
After: (536641, 8)


In [51]:
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]

In [53]:
clean_df["IsCancelled"] = clean_df["InvoiceNo"].astype(str).str.startswith("C")

In [55]:
clean_df["IsReturn"] = clean_df["Quantity"] < 0

In [57]:
customer_df = clean_df.dropna(subset=["CustomerID"]).copy()

In [59]:
clean_df["CustomerID"] = clean_df["CustomerID"].astype("Int64")

In [65]:
clean_df.to_csv("../data/cleaned/online_retail_cleaned.csv", index=False)

customer_df.to_csv("../data/cleaned/customer_transactions.csv", index=False)

In [67]:
import os

print(os.path.exists("../data/cleaned"))
print(os.listdir("../data/cleaned"))

True
['customer_transactions.csv', 'online_retail_cleaned.csv']
